# 3MT–TMC architecture search
Runs **validation-only hyperparameter search** for the completed availability-gated, fixed-equal-fusion 3MT–TMC evidential architecture.

It is kept separate from the final five-fold test evaluation.

## Evaluation protocol
- The prepared outer-fold CSVs, labels, masks, MRI paths, and preprocessing are treated as fixed.
- Hyperparameters are selected using only the `train` and `validation` roles inside a chosen outer fold.
- The `test` role is **never loaded into a DataLoader and never scored in this notebook**.
- Search is performed independently per outer fold so the corresponding held-out test fold remains untouched.
- The fixed 50/50 3MT–TMC fusion rule, evidential formulation, TMC/Dempster–Shafer fusion, cascade order, MRI CNN structure, loss weights, and batch size are held fixed.
- The search changes only a bounded set of capacity, regularisation, and optimiser hyperparameters.
- The existing baseline values are included in every search space.

For a complete nested evaluation, run the notebook separately for folds 0–4 (or set `OUTER_FOLDS_TO_TUNE = [0, 1, 2, 3, 4]` if compute permits), then use each fold's selected configuration only for that fold's final retraining and test evaluation.


In [ ]:
# 1. Environment, reproducibility, paths, and search budget

%pip install -q optuna

from pathlib import Path
import gc
import hashlib
import json
import math
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

warnings.filterwarnings("ignore", category=UserWarning)

# Google Drive

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

MANIFEST_DIR = (
    MODEL_ROOT
    / "manifests"
)

INPUT_DIR = (
    MODEL_ROOT
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

SEARCH_ROOT = (
    MODEL_ROOT
    / "experiments"
    / "architecture_hyperparameter_search_fixed_equal"
)

SEARCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_MODEL_SCHEMA_PATH = (
    MANIFEST_DIR
    / "final_model_input_schema.json"
)

if not FINAL_MODEL_SCHEMA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing prepared model schema: {FINAL_MODEL_SCHEMA_PATH}"
    )

# Search budget

# Tune one outer fold at a time by default so that the test
# partition belonging to that fold remains completely untouched.
OUTER_FOLDS_TO_TUNE = [0]

# 25 trials is a reasonable first search budget for the expensive
# full-volume MRI model. Increase only if the study has not stabilised.
N_TRIALS_PER_FOLD = 25

# Search trials are shorter than final training runs.
SEARCH_MAX_EPOCHS = 30
SEARCH_EARLY_STOPPING_PATIENCE = 7

# The existing scheduler logic is retained.
SCHEDULER_PATIENCE = 3
SCHEDULER_REDUCTION_FACTOR = 0.5
MINIMUM_LEARNING_RATE = 1e-6
MAXIMUM_GRADIENT_NORM = 5.0

# Full MRI volumes keep the batch size fixed at two.
BATCH_SIZE = 2
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# Use the same optimisation seed for every trial within a fold so
# that architectural comparisons are not confounded by a new seed.
SEARCH_SEED = 17

# The final experiment can still be repeated later with seeds
# 17, 42, and 73 after the architecture is selected.
FINAL_CONFIRMATION_SEEDS = [17, 42, 73]

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_MIXED_PRECISION = False

print("=" * 72)
print("ARCHITECTURE HYPERPARAMETER SEARCH")
print("=" * 72)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Outer folds selected for tuning: {OUTER_FOLDS_TO_TUNE}")
print(f"Trials per fold: {N_TRIALS_PER_FOLD}")
print(f"Maximum search epochs per trial: {SEARCH_MAX_EPOCHS}")
print(f"Search output root: {SEARCH_ROOT}")


## 2. Load and verify the prepared schema

The search uses the same prepared fold tables and masks as the completed training pipeline.


In [ ]:
# 2. Load the prepared schema

with open(
    FINAL_MODEL_SCHEMA_PATH,
    "r",
    encoding="utf-8",
) as handle:
    final_model_schema = json.load(handle)

identifier_columns = final_model_schema[
    "identifier_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]

MRI_SPATIAL_SHAPE = (
    177,
    213,
    183,
)

MRI_TENSOR_SHAPE = (
    1,
    *MRI_SPATIAL_SHAPE,
)

print("=" * 72)
print("PREPARED INPUT SCHEMA")
print("=" * 72)
print(f"Target column: {target_column}")
print(f"Scaled continuous features: {len(scaled_continuous_columns)}")
print(f"Encoded categorical features: {len(encoded_categorical_columns)}")
print(f"Branch masks: {len(branch_mask_columns)}")
print(f"Feature masks: {len(feature_mask_columns)}")
print(f"MRI path columns: {len(mri_path_columns)}")
print(f"Modality order: {MODALITY_ORDER}")
print(f"3MT cascade order: {THREE_MT_CASCADE_ORDER}")


## 3. Preserve the branch-level input contract

No new feature engineering or preprocessing is performed.


In [ ]:
# 3. Preserve the multimodal dataset contract

dataset_column_contract = {'demographics': {'continuous': ['DEMOGRAPHICS__AGE_AT_BASELINE__Z',
                                 'DEMOGRAPHICS__PTEDUCAT_CLEAN__Z'],
                  'categorical': ['DEMOGRAPHICS__PTGENDER_CLEAN__IDX',
                                  'DEMOGRAPHICS__PTHAND_CLEAN__IDX'],
                  'feature_masks': ['FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN',
                                    'FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN'],
                  'branch_mask': 'BRANCH_MASK__DEMOGRAPHICS'},
 'cognitive_functional': {'continuous': ['ADAS__TOTSCORE__Z',
                                         'ADAS__TOTAL13__Z',
                                         'MMSE__MMSE_TOTAL_SCORE__Z',
                                         'MMSE__MMSE_ORIENTATION_SCORE__Z',
                                         'MMSE__MMSE_REGISTRATION_SCORE__Z',
                                         'MMSE__MMSE_ATTENTION_SCORE__Z',
                                         'MMSE__MMSE_DELAYED_RECALL_SCORE__Z',
                                         'MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z',
                                         'FAQ__FAQTOTAL__Z'],
                          'categorical': [],
                          'feature_masks': ['FEATURE_MASK__ADAS_TOTSCORE',
                                            'FEATURE_MASK__ADAS_TOTAL13',
                                            'FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE',
                                            'FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE',
                                            'FEATURE_MASK__FAQ_FAQTOTAL'],
                          'branch_mask': 'BRANCH_MASK__COGNITIVE_FUNCTIONAL'},
 'csf': {'continuous': ['CSF__ABETA40__Z',
                        'CSF__ABETA42__Z',
                        'CSF__TAU__Z',
                        'CSF__PTAU__Z',
                        'CSF__ABETA42_40_RATIO__Z'],
         'categorical': [],
         'feature_masks': ['FEATURE_MASK__CSF_ABETA40',
                           'FEATURE_MASK__CSF_ABETA42',
                           'FEATURE_MASK__CSF_TAU',
                           'FEATURE_MASK__CSF_PTAU',
                           'FEATURE_MASK__CSF_ABETA42_40_RATIO'],
         'branch_mask': 'BRANCH_MASK__CSF'},
 'plasma': {'continuous': ['PLASMA__pT217_F__Z',
                           'PLASMA__AB42_F__Z',
                           'PLASMA__AB40_F__Z',
                           'PLASMA__AB42_AB40_F__Z',
                           'PLASMA__pT217_AB42_F__Z',
                           'PLASMA__NfL_Q__Z',
                           'PLASMA__GFAP_Q__Z',
                           'PLASMA__NfL_F__Z',
                           'PLASMA__GFAP_F__Z'],
            'categorical': [],
            'feature_masks': ['FEATURE_MASK__PLASMA_pT217_F',
                              'FEATURE_MASK__PLASMA_AB42_F',
                              'FEATURE_MASK__PLASMA_AB40_F',
                              'FEATURE_MASK__PLASMA_AB42_AB40_F',
                              'FEATURE_MASK__PLASMA_pT217_AB42_F',
                              'FEATURE_MASK__PLASMA_NfL_Q',
                              'FEATURE_MASK__PLASMA_GFAP_Q',
                              'FEATURE_MASK__PLASMA_NfL_F',
                              'FEATURE_MASK__PLASMA_GFAP_F'],
            'branch_mask': 'BRANCH_MASK__PLASMA'},
 'apoe': {'continuous': [],
          'categorical': ['APOE__APOE4_ALLELE_COUNT__IDX'],
          'feature_masks': ['FEATURE_MASK__APOE_APOE4_ALLELE_COUNT'],
          'branch_mask': 'BRANCH_MASK__APOE'},
 'mri': {'path': 'MRI__NORMALIZED_T1_NPY_PATH', 'branch_mask': 'BRANCH_MASK__MRI'}}

print("=" * 72)
print("DATASET CONTRACT")
print("=" * 72)

for branch_name, definition in dataset_column_contract.items():
    print(
        f"{branch_name:24s} | "
        f"continuous={len(definition.get('continuous', [])):2d} | "
        f"categorical={len(definition.get('categorical', [])):2d} | "
        f"feature masks={len(definition.get('feature_masks', [])):2d}"
    )


## 4. Dataset and DataLoader construction

Only the `train` and `validation` roles are converted into DataLoaders.


In [ ]:
# 4. Multimodal dataset and train/validation DataLoader factory

class ADNIMultimodalDataset(Dataset):
    """
    Dataset for the already-prepared ADNI multimodal tables.

    No preprocessing, relabelling, rescaling, or imputation is
    repeated here. MRI arrays are loaded lazily from the prepared
    NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.column_contract = column_contract
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)
        self.target_column = target_column
        self.load_mri = load_mri

    def __len__(self):
        return len(self.dataframe)

    @staticmethod
    def _continuous_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.float32
        )

        return torch.from_numpy(values)

    @staticmethod
    def _categorical_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.int64
        )

        return torch.from_numpy(values)

    @staticmethod
    def _mask_tensor(row, columns):
        if not columns:
            return None

        values = row[columns].to_numpy(
            dtype=np.float32
        )

        return torch.from_numpy(values)

    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1 but the prepared MRI path is missing."
            )

        mri_path = Path(
            str(mri_path)
        )

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array not found: {mri_path}"
            )

        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if tuple(mri_array.shape) != MRI_SPATIAL_SHAPE:
            raise ValueError(
                f"Unexpected MRI shape {mri_array.shape} at {mri_path}"
            )

        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(
            mri_array
        )

    def __getitem__(self, index):
        row = self.dataframe.iloc[
            index
        ]

        modalities = {}

        demographics = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous":
                self._continuous_tensor(
                    row,
                    demographics["continuous"],
                ),
            "categorical":
                self._categorical_tensor(
                    row,
                    demographics["categorical"],
                ),
            "feature_mask":
                self._mask_tensor(
                    row,
                    demographics["feature_masks"],
                ),
            "branch_mask":
                torch.tensor(
                    row[
                        demographics["branch_mask"]
                    ],
                    dtype=torch.float32,
                ),
        }

        for branch_name in [
            "cognitive_functional",
            "csf",
            "plasma",
        ]:
            definition = self.column_contract[
                branch_name
            ]

            modalities[
                branch_name
            ] = {
                "continuous":
                    self._continuous_tensor(
                        row,
                        definition["continuous"],
                    ),
                "feature_mask":
                    self._mask_tensor(
                        row,
                        definition["feature_masks"],
                    ),
                "branch_mask":
                    torch.tensor(
                        row[
                            definition["branch_mask"]
                        ],
                        dtype=torch.float32,
                    ),
            }

        apoe = self.column_contract[
            "apoe"
        ]

        modalities["apoe"] = {
            "categorical":
                self._categorical_tensor(
                    row,
                    apoe["categorical"],
                ),
            "feature_mask":
                self._mask_tensor(
                    row,
                    apoe["feature_masks"],
                ),
            "branch_mask":
                torch.tensor(
                    row[
                        apoe["branch_mask"]
                    ],
                    dtype=torch.float32,
                ),
        }

        mri = self.column_contract[
            "mri"
        ]

        mri_branch_mask = row[
            mri["branch_mask"]
        ]

        modalities["mri"] = {
            "image":
                self._load_mri_tensor(
                    mri_path=row[
                        mri["path"]
                    ],
                    mri_branch_mask=mri_branch_mask,
                ),
            "branch_mask":
                torch.tensor(
                    mri_branch_mask,
                    dtype=torch.float32,
                ),
        }

        return {
            "rid":
                int(row["RID"]),
            "ptid":
                str(row["PTID"]),
            "target":
                torch.tensor(
                    int(
                        row[
                            self.target_column
                        ]
                    ),
                    dtype=torch.long,
                ),
            "modalities":
                modalities,
            "branch_masks":
                self._mask_tensor(
                    row,
                    self.branch_mask_order,
                ),
            "feature_masks":
                self._mask_tensor(
                    row,
                    self.feature_mask_order,
                ),
        }

def set_global_random_seed(seed):
    """
    Seed Python, NumPy, PyTorch, and visible CUDA devices.
    """
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def fold_input_path(
    outer_fold,
):
    return (
        INPUT_DIR
        / f"mci_prognosis_outer_fold_{outer_fold}_final_task_ready.csv"
    )

def load_fold_table(
    outer_fold,
):
    path = fold_input_path(
        outer_fold
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prepared fold input: {path}"
        )

    table = pd.read_csv(
        path
    )

    expected_roles = {
        "train",
        "validation",
        "test",
    }

    observed_roles = set(
        table[
            "DATA_ROLE"
        ].dropna().unique()
    )

    if not expected_roles.issubset(
        observed_roles
    ):
        raise ValueError(
            f"Unexpected role structure in fold {outer_fold}: {observed_roles}"
        )

    return table

def build_train_validation_loaders(
    outer_fold,
    seed,
):
    """
    Build only training and validation loaders.

    Deliberately no test DataLoader is created in this notebook.
    """
    set_global_random_seed(
        seed
    )

    fold_table = load_fold_table(
        outer_fold
    )

    train_table = (
        fold_table.loc[
            fold_table[
                "DATA_ROLE"
            ]
            == "train"
        ]
        .reset_index(
            drop=True
        )
    )

    validation_table = (
        fold_table.loc[
            fold_table[
                "DATA_ROLE"
            ]
            == "validation"
        ]
        .reset_index(
            drop=True
        )
    )

    train_dataset = ADNIMultimodalDataset(
        dataframe=train_table,
        column_contract=dataset_column_contract,
        branch_mask_order=branch_mask_columns,
        feature_mask_order=feature_mask_columns,
        target_column=target_column,
        load_mri=True,
    )

    validation_dataset = ADNIMultimodalDataset(
        dataframe=validation_table,
        column_contract=dataset_column_contract,
        branch_mask_order=branch_mask_columns,
        feature_mask_order=feature_mask_columns,
        target_column=target_column,
        load_mri=True,
    )

    generator = torch.Generator()
    generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    validation_loader = DataLoader(
        dataset=validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    return {
        "train_loader":
            train_loader,
        "validation_loader":
            validation_loader,
        "train_table":
            train_table,
        "validation_table":
            validation_table,
    }

# Structural validation only. The test role is counted but not loaded.
for fold in OUTER_FOLDS_TO_TUNE:
    table = load_fold_table(
        fold
    )

    role_counts = (
        table[
            "DATA_ROLE"
        ]
        .value_counts()
        .to_dict()
    )

    print(
        f"Fold {fold}: "
        f"train={role_counts.get('train', 0)}, "
        f"validation={role_counts.get('validation', 0)}, "
        f"test={role_counts.get('test', 0)} "
        "(test is not loaded or scored)"
    )


## 5. Searchable architecture

The model below preserves the completed six-branch availability-gated design while exposing a small set of architecture hyperparameters.


In [ ]:
# 5. Searchable version of the completed architecture

def _scaled_width(
    base_width,
    scale,
):
    """
    Scale the existing hidden width while keeping widths compact
    and aligned to multiples of eight.
    """
    scaled = int(
        round(
            base_width
            * float(scale)
            / 8.0
        )
        * 8
    )

    return max(
        8,
        scaled,
    )

class MaskAwareScalarEncoder(nn.Module):
    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        input_dim = (
            value_dim
            + mask_dim
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        values,
        feature_mask,
    ):
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class DemographicsEncoder(nn.Module):
    def __init__(
        self,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        input_dim = (
            2
            + 4
            + 4
            + 4
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_representation = (
            self.sex_embedding(
                categorical[
                    :,
                    0,
                ]
            )
        )

        handedness_representation = (
            self.handedness_embedding(
                categorical[
                    :,
                    1,
                ]
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class APOEEncoder(nn.Module):
    def __init__(
        self,
        output_dim,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                9,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
        )

    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_representation = (
            self.apoe_embedding(
                categorical[
                    :,
                    0,
                ]
            )
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(
            inputs
        )

class ResidualDownsampleBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        self.residual_path = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        inputs,
    ):
        pooled = self.pool(
            inputs
        )

        return self.activation(
            self.main_path(
                pooled
            )
            +
            self.residual_path(
                pooled
            )
        )

class MRIEncoder3D(nn.Module):
    """
    The MRI CNN topology, 256-dimensional patch representation,
    one transformer layer, eight MRI attention heads, and 512-unit
    MRI feed-forward layer remain fixed.

    Only the shared projection dimension and dropout scale are tuned.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        dropout=0.20,
    ):
        super().__init__()

        patch_embedding_dim = 256
        transformer_heads = 8
        transformer_layers = 1
        transformer_feedforward_dim = 512

        self.input_shape = tuple(
            input_shape
        )

        self.stem = nn.Sequential(
            nn.Conv3d(
                1,
                16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                16,
                16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                16,
                32,
            ),
            ResidualDownsampleBlock3D(
                32,
                64,
            ),
            ResidualDownsampleBlock3D(
                64,
                128,
            ),
            ResidualDownsampleBlock3D(
                128,
                256,
            ),
        )

        stem_shape = tuple(
            math.ceil(
                dimension
                / 2
            )
            for dimension
            in self.input_shape
        )

        patch_grid_shape = (
            stem_shape
        )

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension
                // 2
                for dimension
                in patch_grid_shape
            )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

    def forward(
        self,
        image,
    ):
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"obtained {actual_patch_count}."
            )

        patch_tokens = (
            feature_map
            .flatten(
                start_dim=2
            )
            .transpose(
                1,
                2,
            )
        )

        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        pooled = transformed_tokens.mean(
            dim=1
        )

        return self.projection(
            pooled
        )

class ADNIModalityEncoders(nn.Module):
    """
    Searchable wrapper around the same six branch encoders.

    At width_scale=1.0 and dropout_scale=1.0 the hidden widths and
    dropout values match the completed fixed-fusion implementation.
    """

    def __init__(
        self,
        output_dim,
        tabular_width_scale,
        dropout_scale,
    ):
        super().__init__()

        scalar_dropout = min(
            0.50,
            0.20
            * dropout_scale,
        )

        apoe_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        mri_dropout = min(
            0.50,
            0.20
            * dropout_scale,
        )

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                48,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.cognitive_functional = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                96,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                64,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                96,
                tabular_width_scale,
            ),
            dropout=scalar_dropout,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
            hidden_dim=_scaled_width(
                32,
                tabular_width_scale,
            ),
            dropout=apoe_dropout,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            dropout=mri_dropout,
        )

    def forward(
        self,
        modalities,
    ):
        return {
            "demographics":
                self.demographics(
                    continuous=
                        modalities[
                            "demographics"
                        ]["continuous"],
                    categorical=
                        modalities[
                            "demographics"
                        ]["categorical"],
                    feature_mask=
                        modalities[
                            "demographics"
                        ]["feature_mask"],
                ),
            "cognitive_functional":
                self.cognitive_functional(
                    values=
                        modalities[
                            "cognitive_functional"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "cognitive_functional"
                        ]["feature_mask"],
                ),
            "csf":
                self.csf(
                    values=
                        modalities[
                            "csf"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "csf"
                        ]["feature_mask"],
                ),
            "plasma":
                self.plasma(
                    values=
                        modalities[
                            "plasma"
                        ]["continuous"],
                    feature_mask=
                        modalities[
                            "plasma"
                        ]["feature_mask"],
                ),
            "apoe":
                self.apoe(
                    categorical=
                        modalities[
                            "apoe"
                        ]["categorical"],
                    feature_mask=
                        modalities[
                            "apoe"
                        ]["feature_mask"],
                ),
            "mri":
                self.mri(
                    modalities[
                        "mri"
                    ]["image"]
                ),
        }

class MaskedADNIModalityEncoders(nn.Module):
    def __init__(
        self,
        output_dim,
        tabular_width_scale,
        dropout_scale,
    ):
        super().__init__()

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
            tabular_width_scale=
                tabular_width_scale,
            dropout_scale=
                dropout_scale,
        )

    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        return (
            representation
            * branch_mask.unsqueeze(
                -1
            )
        )

    def forward(
        self,
        modalities,
    ):
        raw = self.encoders(
            modalities
        )

        masked = {}

        for modality_name in MODALITY_ORDER:
            masked[
                modality_name
            ] = self._apply_branch_mask(
                representation=
                    raw[
                        modality_name
                    ],
                branch_mask=
                    modalities[
                        modality_name
                    ]["branch_mask"],
            )

        return {
            "raw":
                raw,
            "masked":
                masked,
        }

class CascadedModalityTransformer(nn.Module):
    def __init__(
        self,
        embedding_dim,
        number_of_heads,
        dropout,
    ):
        super().__init__()

        if (
            embedding_dim
            % number_of_heads
            != 0
        ):
            raise ValueError(
                "embedding_dim must be divisible by number_of_heads"
            )

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )

    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = (
            self.self_attention_norm(
                latent_query
            )
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = (
            self.cross_attention_norm(
                self_attended_query
            )
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query":
                updated_query,
            "self_attention_weights":
                self_attention_weights,
            "cross_attention_weights":
                cross_attention_weights,
        }

class ThreeMTCascade(nn.Module):
    """
    Availability-gated six-stage 3MT cascade.

    If a branch is absent, the previous query is copied exactly.
    """

    def __init__(
        self,
        embedding_dim,
        modality_order,
        cascade_order,
        number_of_heads,
        dropout,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(
            modality_order
        )
        self.cascade_order = list(
            cascade_order
        )

        self.modality_to_mask_index = {
            modality_name:
                modality_index
            for modality_index, modality_name
            in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=
                            embedding_dim,
                        number_of_heads=
                            number_of_heads,
                        dropout=
                            dropout,
                    )
                for modality_name
                in self.cascade_order
            }
        )

    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = (
            self.cascade_order[
                0
            ]
        )

        batch_size = (
            masked_representations[
                first_modality
            ].shape[
                0
            ]
        )

        latent_query = (
            self.learned_latent_query
            .expand(
                batch_size,
                -1,
                -1,
            )
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = (
                latent_query
            )

            modality_token = (
                masked_representations[
                    modality_name
                ]
                .unsqueeze(
                    1
                )
            )

            stage_output = (
                self.cmt_blocks[
                    modality_name
                ](
                    latent_query=
                        previous_query,
                    modality_embedding=
                        modality_token,
                )
            )

            candidate_query = (
                stage_output[
                    "updated_query"
                ]
            )

            modality_index = (
                self.modality_to_mask_index[
                    modality_name
                ]
            )

            availability = (
                branch_masks[
                    :,
                    modality_index,
                ]
                .view(
                    -1,
                    1,
                    1,
                )
                .to(
                    dtype=
                        previous_query.dtype
                )
            )

            latent_query = (
                availability
                * candidate_query
                +
                (
                    1.0
                    - availability
                )
                * previous_query
            )

            stage_queries[
                modality_name
            ] = latent_query

            self_attention_weights[
                modality_name
            ] = stage_output[
                "self_attention_weights"
            ]

            cross_attention_weights[
                modality_name
            ] = stage_output[
                "cross_attention_weights"
            ]

        return {
            "joint_representation":
                latent_query.squeeze(
                    1
                ),
            "stage_queries":
                stage_queries,
            "self_attention_weights":
                self_attention_weights,
            "cross_attention_weights":
                cross_attention_weights,
        }

class EvidentialClassificationHead(nn.Module):
    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
        branch_mask,
    ):
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(
                -1
            )
        )

        alpha = (
            effective_evidence
            + 1.0
        )

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        return {
            "raw_evidence":
                raw_evidence,
            "evidence":
                effective_evidence,
            "alpha":
                alpha,
            "strength":
                strength,
            "belief":
                effective_evidence
                / strength,
            "probabilities":
                alpha
                / strength,
            "uncertainty":
                self.number_of_classes
                / strength,
        }

class IndependentModalityEvidentialHeads(nn.Module):
    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=
                            input_dim,
                        number_of_classes=
                            number_of_classes,
                        hidden_dim=
                            hidden_dim,
                        dropout=
                            dropout,
                    )
                for modality_name
                in self.modality_order
            }
        )

    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:
            opinions[
                modality_name
            ] = self.heads[
                modality_name
            ](
                representation=
                    modality_representations[
                        modality_name
                    ],
                branch_mask=
                    modalities[
                        modality_name
                    ]["branch_mask"],
            )

        return opinions

class ThreeMTAuxiliaryClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim,
        dropout,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )

    def forward(
        self,
        representation,
    ):
        return self.network(
            representation
        )

class ThreeMTPredictionHeads(nn.Module):
    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
        evidential_hidden_dim,
        dropout,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        self.auxiliary_stages = (
            self.cascade_order[
                :-1
            ]
        )

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=
                            embedding_dim,
                        number_of_classes=
                            number_of_classes,
                        hidden_dim=
                            embedding_dim,
                        dropout=
                            dropout,
                    )
                for stage_name
                in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=
                    embedding_dim,
                number_of_classes=
                    number_of_classes,
                hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    dropout,
            )
        )

    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        for stage_name in self.auxiliary_stages:
            stage_representation = (
                three_mt_output[
                    "stage_queries"
                ][
                    stage_name
                ]
                .squeeze(
                    1
                )
            )

            auxiliary_logits[
                stage_name
            ] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )

        joint_representation = (
            three_mt_output[
                "joint_representation"
            ]
        )

        joint_presence_mask = torch.ones(
            joint_representation.shape[
                0
            ],
            dtype=
                joint_representation.dtype,
            device=
                joint_representation.device,
        )

        joint_opinion = (
            self.joint_evidential_head(
                representation=
                    joint_representation,
                branch_mask=
                    joint_presence_mask,
            )
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,
            "joint_opinion":
                joint_opinion,
        }


## 6. TMC and fixed equal fusion

These components are copied without changing their mathematical rule.


In [ ]:
# 6. Unchanged TMC and fixed-equal fusion modules

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )

    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }

    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]

        # Conflict mass

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)

        # Fused belief and uncertainty masses

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation

        # Recover the fused Dirichlet opinion

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }

    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # Retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }

        # Final fused opinion

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )

class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )

    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )

    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


## 7. Complete model assembly

Every Optuna trial creates a fresh model from the sampled configuration.


In [ ]:
# 7. Complete searchable fixed-equal 3MT–TMC model

class ADNIEvidential3MTTMCModel(nn.Module):
    def __init__(
        self,
        config,
    ):
        super().__init__()

        embedding_dim = int(
            config[
                "embedding_dim"
            ]
        )

        cmt_heads = int(
            config[
                "cmt_heads"
            ]
        )

        tabular_width_scale = float(
            config[
                "tabular_width_scale"
            ]
        )

        dropout_scale = float(
            config[
                "dropout_scale"
            ]
        )

        evidential_hidden_dim = int(
            config[
                "evidential_hidden_dim"
            ]
        )

        modality_dropout_probability = float(
            config[
                "modality_dropout_probability"
            ]
        )

        self.embedding_dim = (
            embedding_dim
        )

        self.number_of_classes = (
            NUMBER_OF_CLASSES
        )

        self.modality_order = list(
            MODALITY_ORDER
        )

        self.cascade_order = list(
            THREE_MT_CASCADE_ORDER
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )

        cmt_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        head_dropout = min(
            0.50,
            0.10
            * dropout_scale,
        )

        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=
                    embedding_dim,
                tabular_width_scale=
                    tabular_width_scale,
                dropout_scale=
                    dropout_scale,
            )
        )

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=
                    self.modality_order,
                input_dim=
                    embedding_dim,
                number_of_classes=
                    NUMBER_OF_CLASSES,
                hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    head_dropout,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=
                NUMBER_OF_CLASSES,
            modality_order=
                self.modality_order,
        )

        self.three_mt_cascade = (
            ThreeMTCascade(
                embedding_dim=
                    embedding_dim,
                modality_order=
                    self.modality_order,
                cascade_order=
                    self.cascade_order,
                number_of_heads=
                    cmt_heads,
                dropout=
                    cmt_dropout,
            )
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=
                    self.cascade_order,
                embedding_dim=
                    embedding_dim,
                number_of_classes=
                    NUMBER_OF_CLASSES,
                evidential_hidden_dim=
                    evidential_hidden_dim,
                dropout=
                    head_dropout,
            )
        )

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=
                    NUMBER_OF_CLASSES,
                number_of_modalities=
                    self.number_of_modalities,
                initial_three_mt_scale=
                    1.0,
                initial_tmc_scale=
                    1.0,
            )
        )

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        if (
            not self.training
            or
            self.modality_dropout_probability
            <= 0.0
        ):
            return (
                original_branch_masks.clone(),
                torch.zeros_like(
                    original_branch_masks
                ),
            )

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=
                    retention_probability,
            )
        )

        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )

        batch_size = (
            original_branch_masks.shape[
                0
            ]
        )

        for batch_row in range(
            batch_size
        ):
            available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ]
                > 0,
                as_tuple=False,
            ).flatten()

            if (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
                and
                available_indices.numel()
                > 0
            ):
                selected_position = torch.randint(
                    low=0,
                    high=
                        available_indices.numel(),
                    size=(1,),
                    device=
                        original_branch_masks.device,
                )

                selected_modality_index = (
                    available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0

        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[
                modality_name
            ] = dict(
                modalities[
                    modality_name
                ]
            )

            effective_modalities[
                modality_name
            ][
                "branch_mask"
            ] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=
                    modalities,
                effective_branch_masks=
                    effective_branch_masks,
            )
        )

        encoded_modalities = (
            self.modality_encoders(
                effective_modalities
            )
        )

        masked_representations = (
            encoded_modalities[
                "masked"
            ]
        )

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=
                    masked_representations,
                modalities=
                    effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=
                modality_opinions
        )

        three_mt_output = (
            self.three_mt_cascade(
                masked_representations=
                    masked_representations,
                branch_masks=
                    effective_branch_masks,
            )
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=
                    three_mt_output
            )
        )

        joint_opinion = (
            three_mt_predictions[
                "joint_opinion"
            ]
        )

        final_output = (
            self.hybrid_fusion(
                joint_opinion=
                    joint_opinion,
                tmc_output=
                    tmc_output,
                branch_masks=
                    effective_branch_masks,
                modality_order=
                    self.modality_order,
            )
        )

        return {
            "final_output":
                final_output,
            "modality_opinions":
                modality_opinions,
            "tmc_output":
                tmc_output,
            "three_mt_output":
                three_mt_output,
            "three_mt_predictions":
                three_mt_predictions,
            "joint_opinion":
                joint_opinion,
            "encoded_modalities":
                encoded_modalities,
            "original_branch_masks":
                original_branch_masks,
            "effective_branch_masks":
                effective_branch_masks,
            "dropped_branch_masks":
                dropped_branch_masks,
        }

def count_trainable_parameters(
    module,
):
    return sum(
        parameter.numel()
        for parameter
        in module.parameters()
        if parameter.requires_grad
    )


## 7.1 Baseline architecture identity check

Before tuning, the check verifies that the searchable implementation reproduces the completed model at the existing baseline settings. The expected trainable-parameter count is **4,428,114**. This check catches accidental architectural changes before any search trial starts.


In [ ]:
# 7.1 Verify that the baseline search configuration reproduces
#     the completed fixed-fusion architecture

BASELINE_ARCHITECTURE_ONLY_CONFIG = {
    "embedding_dim":
        64,
    "cmt_heads":
        4,
    "tabular_width_scale":
        1.0,
    "dropout_scale":
        1.0,
    "evidential_hidden_dim":
        32,
    "modality_dropout_probability":
        0.50,
}

EXPECTED_BASELINE_TRAINABLE_PARAMETERS = (
    4_428_114
)

baseline_identity_model = (
    ADNIEvidential3MTTMCModel(
        config=
            BASELINE_ARCHITECTURE_ONLY_CONFIG
    )
)

observed_baseline_parameters = (
    count_trainable_parameters(
        baseline_identity_model
    )
)

print(
    "Expected trainable parameters: "
    f"{EXPECTED_BASELINE_TRAINABLE_PARAMETERS:,}"
)

print(
    "Observed trainable parameters: "
    f"{observed_baseline_parameters:,}"
)

if (
    observed_baseline_parameters
    !=
    EXPECTED_BASELINE_TRAINABLE_PARAMETERS
):
    raise AssertionError(
        "The searchable baseline does not reproduce the "
        "completed fixed-fusion architecture."
    )

del baseline_identity_model
gc.collect()

print(
    "\nBaseline architecture identity check passed."
)


## 8. Evidential training objective

The loss formulation and component weights are held fixed during the search.


In [ ]:
# 8. Unchanged evidential training objective

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    # Expected cross-entropy under the Dirichlet distribution

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )

    # Optional class weighting

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )

    # Remove correct-class evidence from the KL penalty

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )

    # Requested reduction

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )

    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )

        # Optional training-fold class weights

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )

    # Annealing schedules

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )

    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )

    # Complete loss calculation

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )

        # Final hybrid evidential loss

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]

        # Interaction-aware 3MT evidential loss

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]

        # TMC-fused evidential loss

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]

        # Independent available-modality evidential loss

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )

        # Intermediate 3MT auxiliary cross-entropy loss

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )

        # Early gate-balance regularisation

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )

        # Weighted total objective

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )

        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }

def build_training_objective():
    """
    Preserve the completed fixed-fusion loss formulation and weights.
    """
    return HybridEvidentialTrainingLoss(
        number_of_classes=
            NUMBER_OF_CLASSES,
        modality_order=
            MODALITY_ORDER,
        evidential_annealing_epochs=
            10,
        gate_regularisation_epochs=
            0,
        joint_loss_weight=
            0.50,
        tmc_loss_weight=
            0.50,
        modality_loss_weight=
            0.10,
        auxiliary_loss_weight=
            0.10,
        gate_loss_weight=
            0.0,
        class_weights=
            None,
    )


## 9. Trial training and validation

Validation ROC AUC is the sole optimisation objective. Early stopping and pruning act only on validation performance.


In [ ]:
# 9. Training and validation helpers used by every Optuna trial

def move_nested_to_device(
    value,
    device,
):
    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key:
                move_nested_to_device(
                    item,
                    device,
                )
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_to_device(
                item,
                device,
            )
            for item
            in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_to_device(
                item,
                device,
            )
            for item
            in value
        )

    return value

def run_search_training_epoch(
    model,
    objective,
    loader,
    optimizer,
    epoch,
):
    model.train()

    total_loss = 0.0
    participant_count = 0

    for batch in loader:
        batch = move_nested_to_device(
            batch,
            DEVICE,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=
                batch[
                    "modalities"
                ],
            original_branch_masks=
                batch[
                    "branch_masks"
                ],
        )

        loss_output = objective(
            model_output=
                model_output,
            targets=
                batch[
                    "target"
                ],
            epoch=
                epoch,
        )

        loss = loss_output[
            "total_loss"
        ]

        if not torch.isfinite(
            loss
        ):
            raise FloatingPointError(
                "Non-finite training loss encountered."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=
                MAXIMUM_GRADIENT_NORM,
        )

        optimizer.step()

        batch_size = int(
            batch[
                "target"
            ].shape[
                0
            ]
        )

        total_loss += (
            float(
                loss.detach().cpu()
            )
            * batch_size
        )

        participant_count += (
            batch_size
        )

    return (
        total_loss
        / max(
            participant_count,
            1,
        )
    )

@torch.no_grad()
def evaluate_validation_auc(
    model,
    loader,
):
    model.eval()

    targets = []
    probabilities = []

    for batch in loader:
        batch = move_nested_to_device(
            batch,
            DEVICE,
        )

        model_output = model(
            modalities=
                batch[
                    "modalities"
                ],
            original_branch_masks=
                batch[
                    "branch_masks"
                ],
        )

        p_progressive = (
            model_output[
                "final_output"
            ][
                "probabilities"
            ][
                :,
                1,
            ]
        )

        targets.extend(
            batch[
                "target"
            ]
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        probabilities.extend(
            p_progressive
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if np.unique(
        targets
    ).size < 2:
        raise ValueError(
            "Validation ROC AUC requires both classes."
        )

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )

def fit_search_trial(
    config,
    outer_fold,
    trial=None,
):
    """
    Train one fresh model on one outer fold's training role and
    select epochs using only that fold's validation role.
    """
    set_global_random_seed(
        SEARCH_SEED
    )

    loaders = (
        build_train_validation_loaders(
            outer_fold=
                outer_fold,
            seed=
                SEARCH_SEED,
        )
    )

    model = ADNIEvidential3MTTMCModel(
        config=
            config
    ).to(
        DEVICE
    )

    objective = (
        build_training_objective()
        .to(
            DEVICE
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=
            float(
                config[
                    "learning_rate"
                ]
            ),
        weight_decay=
            float(
                config[
                    "weight_decay"
                ]
            ),
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=
                SCHEDULER_REDUCTION_FACTOR,
            patience=
                SCHEDULER_PATIENCE,
            min_lr=
                MINIMUM_LEARNING_RATE,
        )
    )

    best_validation_auc = (
        -np.inf
    )

    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    for epoch in range(
        1,
        SEARCH_MAX_EPOCHS
        + 1,
    ):
        train_loss = (
            run_search_training_epoch(
                model=
                    model,
                objective=
                    objective,
                loader=
                    loaders[
                        "train_loader"
                    ],
                optimizer=
                    optimizer,
                epoch=
                    epoch,
            )
        )

        validation_auc = (
            evaluate_validation_auc(
                model=
                    model,
                loader=
                    loaders[
                        "validation_loader"
                    ],
            )
        )

        scheduler.step(
            validation_auc
        )

        current_lr = float(
            optimizer.param_groups[
                0
            ]["lr"]
        )

        history.append(
            {
                "epoch":
                    epoch,
                "train_loss":
                    train_loss,
                "validation_auc":
                    validation_auc,
                "learning_rate":
                    current_lr,
            }
        )

        if validation_auc > (
            best_validation_auc
            + 1e-6
        ):
            best_validation_auc = (
                validation_auc
            )
            best_epoch = (
                epoch
            )
            epochs_without_improvement = (
                0
            )
        else:
            epochs_without_improvement += (
                1
            )

        if trial is not None:
            trial.report(
                validation_auc,
                step=
                    epoch,
            )

            if trial.should_prune():
                del model
                del objective
                del optimizer

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                raise optuna.TrialPruned(
                    f"Pruned at epoch {epoch}"
                )

        if (
            epochs_without_improvement
            >=
            SEARCH_EARLY_STOPPING_PATIENCE
        ):
            break

    parameter_count = (
        count_trainable_parameters(
            model
        )
    )

    del model
    del objective
    del optimizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "best_validation_auc":
            float(
                best_validation_auc
            ),
        "best_epoch":
            int(
                best_epoch
            ),
        "parameter_count":
            int(
                parameter_count
            ),
        "history":
            history,
    }


## 10. Search space

The search is deliberately bounded. It includes the current model's baseline configuration exactly:

| Hyperparameter | Search range | Existing value |
|---|---|---:|
| Shared modality / CMT dimension | 32, 64, 128 | 64 |
| CMT attention heads | 2, 4, 8 | 4 |
| Tabular hidden-width scale | 0.75, 1.0, 1.25 | 1.0 |
| Global dropout scale | 0.5, 1.0, 1.5 | 1.0 |
| Evidential-head hidden dimension | 16, 32, 64 | 32 |
| Training-time modality dropout | 0.25, 0.50, 0.65 | 0.50 |
| AdamW learning rate | log-uniform 1e-4 to 1e-3 | 5e-4 |
| AdamW weight decay | log-uniform 1e-6 to 1e-3 | 1e-4 |

The MRI convolutional channel sequence, MRI patch dimension, MRI transformer depth, TMC rule, fixed equal fusion, evidential loss weights, cascade order, batch size, and prepared data remain unchanged. This keeps the experiment focused on capacity and regularisation rather than redefining the method.


In [ ]:
# 10. Optuna objective and resumable per-fold studies

BASELINE_SEARCH_CONFIG = {
    "embedding_dim":
        64,
    "cmt_heads":
        4,
    "tabular_width_scale":
        1.0,
    "dropout_scale":
        1.0,
    "evidential_hidden_dim":
        32,
    "modality_dropout_probability":
        0.50,
    "learning_rate":
        5e-4,
    "weight_decay":
        1e-4,
}

def sample_search_config(
    trial,
):
    return {
        "embedding_dim":
            trial.suggest_categorical(
                "embedding_dim",
                [
                    32,
                    64,
                    128,
                ],
            ),

        "cmt_heads":
            trial.suggest_categorical(
                "cmt_heads",
                [
                    2,
                    4,
                    8,
                ],
            ),

        "tabular_width_scale":
            trial.suggest_categorical(
                "tabular_width_scale",
                [
                    0.75,
                    1.0,
                    1.25,
                ],
            ),

        "dropout_scale":
            trial.suggest_categorical(
                "dropout_scale",
                [
                    0.5,
                    1.0,
                    1.5,
                ],
            ),

        "evidential_hidden_dim":
            trial.suggest_categorical(
                "evidential_hidden_dim",
                [
                    16,
                    32,
                    64,
                ],
            ),

        "modality_dropout_probability":
            trial.suggest_categorical(
                "modality_dropout_probability",
                [
                    0.25,
                    0.50,
                    0.65,
                ],
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-4,
                1e-3,
                log=True,
            ),

        "weight_decay":
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-3,
                log=True,
            ),
    }

def make_objective(
    outer_fold,
):
    def objective(
        trial,
    ):
        config = sample_search_config(
            trial
        )

        try:
            result = fit_search_trial(
                config=
                    config,
                outer_fold=
                    outer_fold,
                trial=
                    trial,
            )

        except torch.cuda.OutOfMemoryError:
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            raise optuna.TrialPruned(
                "CUDA out of memory"
            )

        trial.set_user_attr(
            "best_epoch",
            result[
                "best_epoch"
            ],
        )

        trial.set_user_attr(
            "parameter_count",
            result[
                "parameter_count"
            ],
        )

        return result[
            "best_validation_auc"
        ]

    return objective

def study_paths(
    outer_fold,
):
    fold_dir = (
        SEARCH_ROOT
        / f"fold_{outer_fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "fold_dir":
            fold_dir,
        "database":
            fold_dir
            / "optuna_study.sqlite3",
        "trials_csv":
            fold_dir
            / "trials.csv",
        "best_params":
            fold_dir
            / "best_params.json",
        "study_summary":
            fold_dir
            / "study_summary.json",
    }

def run_or_resume_fold_search(
    outer_fold,
):
    paths = study_paths(
        outer_fold
    )

    storage_url = (
        "sqlite:///"
        + str(
            paths[
                "database"
            ]
        )
    )

    sampler = TPESampler(
        seed=
            SEARCH_SEED,
        multivariate=True,
    )

    pruner = MedianPruner(
        n_startup_trials=
            5,
        n_warmup_steps=
            5,
        interval_steps=
            1,
    )

    study = optuna.create_study(
        study_name=
            f"architecture_search_fold_{outer_fold}",
        direction=
            "maximize",
        sampler=
            sampler,
        pruner=
            pruner,
        storage=
            storage_url,
        load_if_exists=
            True,
    )

    # Queue the completed experiment's configuration first so
    # the search always has a direct baseline inside the same study.
    if len(
        study.trials
    ) == 0:
        study.enqueue_trial(
            BASELINE_SEARCH_CONFIG
        )

    already_finished = sum(
        trial.state
        in {
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
        }
        for trial
        in study.trials
    )

    remaining_trials = max(
        0,
        N_TRIALS_PER_FOLD
        - already_finished,
    )

    print("=" * 72)
    print(
        f"OUTER FOLD {outer_fold} SEARCH"
    )
    print("=" * 72)
    print(
        f"Existing finished trials: {already_finished}"
    )
    print(
        f"Trials still requested: {remaining_trials}"
    )
    print(
        f"Study database: {paths['database']}"
    )

    if remaining_trials > 0:
        study.optimize(
            make_objective(
                outer_fold
            ),
            n_trials=
                remaining_trials,
            gc_after_trial=
                True,
            show_progress_bar=
                True,
        )

    trials_df = (
        study.trials_dataframe(
            attrs=(
                "number",
                "value",
                "state",
                "params",
                "user_attrs",
            )
        )
    )

    trials_df.to_csv(
        paths[
            "trials_csv"
        ],
        index=False,
    )

    best_payload = {
        "outer_fold":
            int(
                outer_fold
            ),
        "objective":
            "best validation ROC AUC",
        "best_trial_number":
            int(
                study.best_trial.number
            ),
        "best_validation_auc":
            float(
                study.best_value
            ),
        "best_params":
            {
                key:
                    (
                        value.item()
                        if isinstance(
                            value,
                            np.generic,
                        )
                        else value
                    )
                for key, value
                in study.best_params.items()
            },
        "best_epoch":
            int(
                study.best_trial.user_attrs.get(
                    "best_epoch",
                    -1,
                )
            ),
        "parameter_count":
            int(
                study.best_trial.user_attrs.get(
                    "parameter_count",
                    -1,
                )
            ),
        "search_seed":
            int(
                SEARCH_SEED
            ),
        "n_trials_requested":
            int(
                N_TRIALS_PER_FOLD
            ),
        "test_data_used":
            False,
    }

    with open(
        paths[
            "best_params"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            best_payload,
            handle,
            indent=2,
        )

    study_summary = {
        "source_architecture":
            "availability-gated fixed-equal 3MT-TMC evidential model",
        "fold":
            int(
                outer_fold
            ),
        "train_role_used":
            True,
        "validation_role_used":
            True,
        "test_role_loaded":
            False,
        "test_role_scored":
            False,
        "batch_size":
            int(
                BATCH_SIZE
            ),
        "search_max_epochs":
            int(
                SEARCH_MAX_EPOCHS
            ),
        "early_stopping_patience":
            int(
                SEARCH_EARLY_STOPPING_PATIENCE
            ),
        "scheduler_patience":
            int(
                SCHEDULER_PATIENCE
            ),
        "baseline_config":
            BASELINE_SEARCH_CONFIG,
        "best_result":
            best_payload,
    }

    with open(
        paths[
            "study_summary"
        ],
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            study_summary,
            handle,
            indent=2,
        )

    return (
        study,
        trials_df,
        best_payload,
    )


## 11. Smoke test

This cell runs **one training epoch and one validation pass** using the existing baseline configuration on the first selected tuning fold. It is useful before committing hours to the Optuna search.

It still does not load or score the test role.


In [ ]:
# 11. One-epoch baseline smoke test

RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_fold = int(
        OUTER_FOLDS_TO_TUNE[
            0
        ]
    )

    original_max_epochs = (
        SEARCH_MAX_EPOCHS
    )

    # Run the same trial helper with one epoch by temporarily
    # shortening the search horizon.
    SEARCH_MAX_EPOCHS = 1

    smoke_result = fit_search_trial(
        config=
            BASELINE_SEARCH_CONFIG,
        outer_fold=
            smoke_fold,
        trial=
            None,
    )

    SEARCH_MAX_EPOCHS = (
        original_max_epochs
    )

    print("=" * 72)
    print("SMOKE TEST COMPLETE")
    print("=" * 72)
    print(
        f"Fold: {smoke_fold}"
    )
    print(
        "Validation ROC AUC after one epoch: "
        f"{smoke_result['best_validation_auc']:.4f}"
    )
    print(
        "Trainable parameters: "
        f"{smoke_result['parameter_count']:,}"
    )


## 12. Run the search

The SQLite study is stored on Drive, so rerunning the notebook resumes completed work instead of starting over.


In [ ]:
# 12. Run or resume the requested per-fold hyperparameter searches

all_studies = {}
all_best_results = []

for outer_fold in OUTER_FOLDS_TO_TUNE:
    (
        study,
        trials_df,
        best_payload,
    ) = run_or_resume_fold_search(
        int(
            outer_fold
        )
    )

    all_studies[
        int(
            outer_fold
        )
    ] = study

    all_best_results.append(
        best_payload
    )

best_results_table = pd.DataFrame(
    [
        {
            "OUTER_FOLD":
                item[
                    "outer_fold"
                ],
            "BEST_TRIAL":
                item[
                    "best_trial_number"
                ],
            "VALIDATION_ROC_AUC":
                item[
                    "best_validation_auc"
                ],
            "BEST_EPOCH":
                item[
                    "best_epoch"
                ],
            "PARAMETER_COUNT":
                item[
                    "parameter_count"
                ],
            **{
                f"PARAM__{key}":
                    value
                for key, value
                in item[
                    "best_params"
                ].items()
            },
        }
        for item
        in all_best_results
    ]
)

display(
    best_results_table
)

aggregate_path = (
    SEARCH_ROOT
    / "best_results_selected_folds.csv"
)

best_results_table.to_csv(
    aggregate_path,
    index=False,
)

print(
    f"\nSaved summary: {aggregate_path}"
)


## 13. Review the best trials

This compares the best observed validation result against the enqueued baseline configuration inside the same fold-specific study.


In [ ]:
# 13. Inspect completed trials for each tuned fold

for outer_fold, study in all_studies.items():
    print("\n" + "=" * 72)
    print(
        f"FOLD {outer_fold} — TOP COMPLETED TRIALS"
    )
    print("=" * 72)

    completed = [
        trial
        for trial
        in study.trials
        if trial.state
        ==
        optuna.trial.TrialState.COMPLETE
    ]

    completed = sorted(
        completed,
        key=lambda trial:
            trial.value,
        reverse=True,
    )

    top_rows = []

    for trial in completed[
        :10
    ]:
        row = {
            "TRIAL":
                trial.number,
            "VALIDATION_ROC_AUC":
                trial.value,
            "BEST_EPOCH":
                trial.user_attrs.get(
                    "best_epoch"
                ),
            "PARAMETER_COUNT":
                trial.user_attrs.get(
                    "parameter_count"
                ),
        }

        row.update(
            trial.params
        )

        top_rows.append(
            row
        )

    display(
        pd.DataFrame(
            top_rows
        )
    )

    baseline_trials = [
        trial
        for trial
        in completed
        if all(
            trial.params.get(
                key
            )
            ==
            value
            for key, value
            in BASELINE_SEARCH_CONFIG.items()
        )
    ]

    if baseline_trials:
        baseline_trial = (
            baseline_trials[
                0
            ]
        )

        print(
            "Baseline validation ROC AUC: "
            f"{baseline_trial.value:.4f}"
        )

        print(
            "Best validation ROC AUC: "
            f"{study.best_value:.4f}"
        )

        print(
            "Difference: "
            f"{study.best_value - baseline_trial.value:+.4f}"
        )


## 14. Outputs

For each outer fold, the notebook saves:

- `optuna_study.sqlite3` — resumable Optuna study;
- `trials.csv` — every completed/pruned/failed trial and its parameters;
- `best_params.json` — the selected configuration for that outer fold;
- `study_summary.json` — audit record showing that only training and validation roles were used.

The search notebook deliberately stops here. It **does not evaluate the test role**.

For a strict nested-CV result, use `fold_0/best_params.json` only when retraining/evaluating outer fold 0, `fold_1/best_params.json` only for outer fold 1, and so on. After all five folds are tuned and evaluated independently, aggregate the five untouched test predictions exactly as in the existing cross-validation analysis.

For a single deployment configuration rather than an unbiased performance estimate, the per-fold selected configurations can be compared after nested evaluation and a final model can be trained on all available development data.
